# ML 08a — Baseline Models

This notebook establishes two non-ML reference points using the frozen chronological split: a constant training-median baseline and a hierarchical historical-lag baseline. The 2024 test target remains sealed.

## Safeguards

- Uses the same three expanding-window folds and 2023 validation period as every ML experiment.
- Historical fallback order: exact group lag 1 → species lag 1 → cell lag 1 → global lag 1 → training-target median.
- Reports identical log-scale and kilogram-scale metrics used by later models.
- Does not fit, predict or score any 2024 test row.

In [ ]:
import importlib.util, subprocess, sys
packages={'pandas':'pandas>=2.0,<3','numpy':'numpy>=1.24,<3','pyarrow':'pyarrow>=14,<22','matplotlib':'matplotlib>=3.7,<4','sklearn':'scikit-learn>=1.3,<2'}
missing=[spec for module,spec in packages.items() if importlib.util.find_spec(module) is None]
if missing: subprocess.check_call([sys.executable,'-m','pip','install',*missing])
print('Baseline dependencies are available.')

In [ ]:
from __future__ import annotations
import hashlib, json, os, sys
from pathlib import Path
import matplotlib
if 'IPython' not in sys.modules: matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error, r2_score
try:
    from IPython.display import display
except ImportError:
    display=print
plt.style.use('seaborn-v0_8-whitegrid')
EXPERIMENT='baselines'

In [ ]:
def find_component_root(start:Path)->Path:
    for anchor in [start.resolve(),*start.resolve().parents]:
        for candidate in [anchor,anchor/'component - 3',anchor/'J26-DS-350'/'component - 3']:
            if (candidate/'Notebooks').is_dir() and (candidate/'src').is_dir(): return candidate.resolve()
    raise FileNotFoundError('Component 3 repository root was not found.')
IN_COLAB='google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive'); ROOT=Path('/content/drive/MyDrive/FishGuard-C3')
else: ROOT=find_component_root(Path.cwd())
DATA_PATH=ROOT/'data'/'processed'/'monthly_model_features_split.parquet'
FOLDS_PATH=ROOT/'outputs'/'audit'/'modelling_eda_split'/'expanding_window_cv_folds.csv'
EVIDENCE_DIR=ROOT/'outputs'/'audit'/'model_training'/EXPERIMENT
METRIC_DIR=ROOT/'outputs'/'metrics'/'model_training'/EXPERIMENT
PREDICTION_DIR=ROOT/'outputs'/'predictions'/'model_training'/EXPERIMENT
FIGURE_DIR=ROOT/'outputs'/'figures'/'model_training'/EXPERIMENT
for directory in [EVIDENCE_DIR,METRIC_DIR,PREDICTION_DIR,FIGURE_DIR]: directory.mkdir(parents=True,exist_ok=True)
if not DATA_PATH.is_file() or not FOLDS_PATH.is_file(): raise FileNotFoundError('Run 07_modelling_eda_and_split.ipynb first.')
data=pd.read_parquet(DATA_PATH); data['month_start']=pd.to_datetime(data.month_start); folds=pd.read_csv(FOLDS_PATH)
TARGET='target_log1p_median_catch'
train=data.loc[data.data_split.astype(str)=='train'].copy(); validation=data.loc[data.data_split.astype(str)=='validation'].copy()
test_rows=int((data.data_split.astype(str)=='test').sum()); source_sha256=hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
print('Train:',len(train),'Validation:',len(validation),'Sealed test:',test_rows)

In [ ]:
def metrics(y_true_log,y_pred_log):
    y_true_log=np.asarray(y_true_log,float); y_pred_log=np.asarray(y_pred_log,float)
    true_kg=np.clip(np.expm1(y_true_log),0,None); pred_kg=np.clip(np.expm1(np.clip(y_pred_log,0,None)),0,None)
    return {'log_mae':mean_absolute_error(y_true_log,y_pred_log),'log_rmse':mean_squared_error(y_true_log,y_pred_log)**.5,'log_median_ae':median_absolute_error(y_true_log,y_pred_log),'log_r2':r2_score(y_true_log,y_pred_log),'kg_mae':mean_absolute_error(true_kg,pred_kg),'kg_rmse':mean_squared_error(true_kg,pred_kg)**.5,'kg_median_ae':median_absolute_error(true_kg,pred_kg),'kg_bias':float(np.mean(pred_kg-true_kg))}
def historical(frame,fallback):
    values=frame.group_median_catch_weight_kg_lag_1.combine_first(frame.species_median_catch_weight_kg_lag_1).combine_first(frame.cell_median_catch_weight_kg_lag_1).combine_first(frame.global_median_catch_weight_kg_lag_1)
    return np.log1p(values.clip(lower=0)).fillna(fallback).to_numpy()
rows=[]
for fold in folds.itertuples(index=False):
    fold_train=train.loc[train.month_start<=pd.Timestamp(fold.train_end)]; fold_valid=train.loc[(train.month_start>=pd.Timestamp(fold.validation_start))&(train.month_start<=pd.Timestamp(fold.validation_end))]
    fallback=float(fold_train[TARGET].median())
    predictions={'median_baseline':np.full(len(fold_valid),fallback),'historical_baseline':historical(fold_valid,fallback)}
    for model,prediction in predictions.items(): rows.append({'fold_id':int(fold.fold_id),'model':model,'train_rows':len(fold_train),'validation_rows':len(fold_valid),**metrics(fold_valid[TARGET],prediction)})
cv=pd.DataFrame(rows)
cv.to_csv(METRIC_DIR/'cv_metrics.csv',index=False)
display(cv)

In [ ]:
fallback=float(train[TARGET].median())
validation_prediction_map={'median_baseline':np.full(len(validation),fallback),'historical_baseline':historical(validation,fallback)}
validation_metrics=[]; prediction_rows=[]
for model,prediction in validation_prediction_map.items():
    validation_metrics.append({'model':model,**metrics(validation[TARGET],prediction)})
    for index,value in zip(validation.index,prediction): prediction_rows.append({'row_index':int(index),'model':model,'prediction_log':value})
validation_metrics=pd.DataFrame(validation_metrics).sort_values('log_mae')
keys=validation[['month_start','spatial_cell_id','species_model','gear_model',TARGET,'target_median_catch_weight_kg']].reset_index().rename(columns={'index':'row_index','target_median_catch_weight_kg':'actual_kg'})
predictions=pd.DataFrame(prediction_rows).merge(keys,on='row_index',how='left',validate='m:1')
predictions['prediction_kg']=np.expm1(predictions.prediction_log.clip(lower=0))
validation_metrics.to_csv(METRIC_DIR/'validation_metrics.csv',index=False); predictions.to_csv(PREDICTION_DIR/'validation_predictions.csv',index=False)
summary=cv.groupby('model').agg(cv_log_mae_mean=('log_mae','mean'),cv_log_mae_std=('log_mae','std'),cv_kg_mae_mean=('kg_mae','mean')).reset_index().merge(validation_metrics,on='model',validate='1:1')
summary.to_csv(METRIC_DIR/'experiment_summary.csv',index=False)
display(summary)

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(13,5))
for model,color in [('median_baseline','#bab0ac'),('historical_baseline','#4e79a7')]:
    part=cv.loc[cv.model==model]; axes[0].plot(part.fold_id,part.log_mae,marker='o',label=model,color=color)
axes[0].set_xticks(sorted(cv.fold_id.unique())); axes[0].set_xlabel('Expanding-window fold'); axes[0].set_ylabel('Log MAE'); axes[0].set_title('Cross-validation baselines'); axes[0].legend()
axes[1].barh(validation_metrics.model,validation_metrics.log_mae,color=['#4e79a7' if name=='historical_baseline' else '#bab0ac' for name in validation_metrics.model]); axes[1].set_xlabel('2023 validation log MAE'); axes[1].set_title('Validation baselines')
fig.tight_layout(); fig.savefig(FIGURE_DIR/'baseline_comparison.png',dpi=200,bbox_inches='tight'); plt.show()
checks=pd.DataFrame([
 {'check':'test excluded','passed':len(train)+len(validation)+test_rows==len(data),'detail':f'{test_rows} sealed rows'},
 {'check':'three chronological folds','passed':cv.fold_id.nunique()==3,'detail':str(sorted(cv.fold_id.unique()))},
 {'check':'finite predictions','passed':np.isfinite(predictions.prediction_log).all(),'detail':f'{len(predictions)} predictions'}])
assert checks.passed.all()
checks.to_csv(EVIDENCE_DIR/'baseline_checks.csv',index=False)
best=summary.sort_values('log_mae').iloc[0]
findings=pd.DataFrame([{'finding':'Best validation baseline','result':str(best.model)},{'finding':'Best baseline validation log MAE','result':f'{best.log_mae:.4f}'},{'finding':'Test status','result':'2024 target sealed and unused'}])
findings.to_csv(EVIDENCE_DIR/'baseline_findings.csv',index=False)
manifest={'created_utc':pd.Timestamp.now(tz='UTC').isoformat(),'runtime':'Google Colab' if IN_COLAB else 'Local Windows','experiment':EXPERIMENT,'input':DATA_PATH.name,'input_sha256':source_sha256,'train_rows':len(train),'validation_rows':len(validation),'sealed_test_rows':test_rows,'test_target_used':False,'models':['median_baseline','historical_baseline'],'metrics':sorted(p.name for p in METRIC_DIR.glob('*.csv')),'predictions':sorted(p.name for p in PREDICTION_DIR.glob('*.csv')),'figures':sorted(p.name for p in FIGURE_DIR.glob('*.png'))}
with (EVIDENCE_DIR/'baseline_manifest.json').open('w',encoding='utf-8') as f: json.dump(manifest,f,indent=2)
display(checks); display(findings)

## Viva interpretation

A trained ML model must outperform useful reference strategies. The median baseline tests whether learning adds value beyond a constant forecast; the hierarchical historical baseline tests whether ML adds value beyond simply carrying forward the latest available catch information.